In [4]:
from utils.build_data import get_data
from utils import eval, spike, data_processing
from utils.constants import (
    TARGET_COLUMN,
    RANDOM_STATE
)

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import RandomOverSampler
import os

In [5]:
COMMODITYS = [
    'cobalt', 
    'copper', 
    'magnesium', 
    'nickel',
]

#target_COMMODITY = "copper"
# target_COMMODITY = "cobalt"
WINDOW_SIZE = 20

pre_features = []
pre_labels = []
tar_features = []
tar_labels = []

In [6]:
for target_COMMODITY in COMMODITYS:

    # for COMMODITY in COMMODITYS:
    VOLZA_FILE_PATH = f"../../datasets/{target_COMMODITY}/{target_COMMODITY}.csv"
    PRICE_FILE_PATH = f"../../datasets/{target_COMMODITY}/{target_COMMODITY}_prices.csv"

    # Get the data
    data = get_data(VOLZA_FILE_PATH, PRICE_FILE_PATH, window_size=WINDOW_SIZE, center=False)

    # Add spike column
    data['spikes_streaming'] = spike.detect_spikes_shift(data, 'Price', window_size=WINDOW_SIZE)

    sampler = RandomOverSampler
    keyword = 'shift'

    # Evaluate and create pre-trained model
    RUN_ID = f"{target_COMMODITY}_{keyword}_{WINDOW_SIZE}d"
    BASE_DIR = f"model_results/{RUN_ID}"
    os.makedirs(BASE_DIR, exist_ok=True)

    METRICS_PATH = f"{BASE_DIR}/metrics.csv"
    PREDICTIONS_DIR = f"{BASE_DIR}/predictions/test"
    MODELS_DIR = f"{BASE_DIR}/models/"

    print(PREDICTIONS_DIR)


    # Prepare price data
    X_price, y_price = data_processing.prepare_features_and_target(data, TARGET_COLUMN, 'spikes_streaming')
    # X_price = np.array(tar_features)
    # y_price = np.array(tar_labels)

    # Split price data
    X_train_price, X_test_price, y_train_price, y_test_price = train_test_split(X_price, y_price, test_size=0.4, shuffle=False)
    X_train_price, y_train_price = RandomOverSampler(random_state=RANDOM_STATE).fit_resample(X_train_price, y_train_price)

    # Balancing
    X_train_price, y_train_price = sampler(random_state=RANDOM_STATE).fit_resample(X_train_price, y_train_price)

    # Scaling
    X_train_price, X_test_price = data_processing.scale_features_no_val(X_train_price, X_test_price)

    # Sequence making
    X_train_price, y_train_price = data_processing.create_sequences(X_train_price, y_train_price, WINDOW_SIZE)
    X_test_price, y_test_price = data_processing.create_sequences(X_test_price, y_test_price, WINDOW_SIZE)
    # X_val_price, y_val_price = data_processing.create_sequences(X_val_price, y_val_price, WINDOW_SIZE)

    # Use this for Bowen's method
    # X_train_price = np.expand_dims(X_train_price, axis = 2)
    # X_test_price = np.expand_dims(X_test_price, axis = 2)

    results_df  = eval.evaluate_all(X_train_price, y_train_price, None, None, X_test_price, y_test_price, METRICS_PATH, PREDICTIONS_DIR, MODELS_DIR, False, val=False)

Building data...
Rows dropped due to NaN values: 1


c:\Users\manim\LabV2\onr_price_prediction\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\manim\LabV2\onr_price_prediction\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\manim\LabV2\onr_price_prediction\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


model_results/cobalt_shift_20d/predictions/test
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step


Predictions saved to CSV file: model_results/cobalt_shift_20d/predictions/test/LSTM_256_layers_predictions.csv
Predictions saved to CSV file: model_results/cobalt_shift_20d/predictions/test_confidence/LSTM_256_layers_predictions.csv
{'Name': 'LSTM', 'Params': '256 layers', 'Accuracy': 0.8452380952380952, 'Precision (0)': 0.848, 'Recall (0)': 0.9953051643192489, 'F1 (0)': 0.9157667386609071, 'Precision (1)': 0.5, 'Recall (1)': 0.02564102564102564, 'F1 (1)': 0.04878048780487805, 'Prior': '0.15'}


c:\Users\manim\LabV2\onr_price_prediction\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

c:\Users\manim\LabV2\onr_price_prediction\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\manim\LabV2\onr_price_prediction\venv\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\manim\LabV2\onr_price_prediction\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


model_results/magnesium_shift_20d/predictions/test
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step


Predictions saved to CSV file: model_results/magnesium_shift_20d/predictions/test/LSTM_256_layers_predictions.csv
Predictions saved to CSV file: model_results/magnesium_shift_20d/predictions/test_confidence/LSTM_256_layers_predictions.csv
{'Name': 'LSTM', 'Params': '256 layers', 'Accuracy': 0.6449864498644986, 'Precision (0)': 0.8223938223938224, 'Recall (0)': 0.714765100671141, 'F1 (0)': 0.7648114901256733, 'Precision (1)': 0.22727272727272727, 'Recall (1)': 0.352112676056338, 'F1 (1)': 0.27624309392265195, 'Prior': '0.19'}


c:\Users\manim\LabV2\onr_price_prediction\venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


KeyboardInterrupt: 